# Deterministic ensembles with noise conditioning

Train a noise-conditioned ViT with Lightning and CRPS, then inspect individual members, the ensemble mean and spread. This notebook runs independently: it uses physical fields, with no autoencoder or cached latents.

Unlike diffusion and flow matching, each forecast needs one forward pass per member, not an iterative sampler. The network is deterministic given its inputs and noise; different noise vectors produce different members.

In [ ]:
import lightning as L
import matplotlib.pyplot as plt
import torch
from _support import OUTPUT_ROOT
from omegaconf import OmegaConf

from autocast.data.utils import get_autosim_datamodule
from autocast.decoders.channels_last import ChannelsLast
from autocast.encoders.permute_concat import PermuteConcat
from autocast.losses.ensemble import CRPSLoss
from autocast.models.encoder_decoder import EncoderDecoder
from autocast.models.encoder_processor_decoder_ensemble import (
    EncoderProcessorDecoderEnsemble,
)
from autocast.processors.azula_vit import AzulaViTProcessor
from autocast.utils import get_optimizer_config
from autocast.utils.plots import plot_spatiotemporal_snapshots

torch.set_num_threads(2)
L.seed_everything(42, workers=True)

## Generate the data

Generate the same small, seeded AutoSim splits used by the autoencoder tutorial, in this run's own directory. Keep fields in physical units and predict two frames from the preceding two.

In [ ]:
run_dir = OUTPUT_ROOT / "deterministic_ensemble"
data_options = {
    "data_path": str(run_dir / "data"),
    "n_steps_input": 2,
    "n_steps_output": 2,
    "stride": 2,
    "batch_size": 4,
    "num_workers": 0,
    "use_normalization": False,
}
datamodule = get_autosim_datamodule(
    simulation_name="advection_diffusion",
    simulator_kwargs={"n": 16, "T": 1.0, "dt": 0.1},
    n_train=6,
    n_valid=2,
    n_test=2,
    overwrite=True,
    seed=42,
    **data_options,
)
batch = next(iter(datamodule.train_dataloader()))
batch.input_fields.shape, batch.output_fields.shape

## Compose the ensemble

As in the [quickstart](quickstart.ipynb), the encoder and decoder only rearrange tensors. The ViT receives two frames folded into two channels and uses a noise vector to modulate its layers—not to perturb the input fields.

In [ ]:
encoder_options = {"in_channels": 1, "n_steps_input": 2}
decoder_options = {"output_channels": 1, "time_steps": 2}
encoder_decoder = EncoderDecoder(
    encoder=PermuteConcat(**encoder_options),
    decoder=ChannelsLast(**decoder_options),
)
noise_channels = 16
processor_options = {
    "in_channels": 2,
    "out_channels": 2,
    "spatial_resolution": (16, 16),
    "hidden_dim": 32,
    "num_heads": 4,
    "n_layers": 2,
    "patch_size": 4,
    "temporal_method": "none",
    "n_noise_channels": noise_channels,
    "global_cond_channels": batch.constant_scalars.shape[-1],
    "include_global_cond": True,
}
processor = AzulaViTProcessor(**processor_options)

`EncoderProcessorDecoderEnsemble` repeats each input for its members and returns a final member axis. `CRPSLoss` scores those members jointly against the observed fields.

In [ ]:
training_members = 3
optimizer = get_optimizer_config(learning_rate=3e-3)
model = EncoderProcessorDecoderEnsemble(
    encoder_decoder=encoder_decoder,
    processor=processor,
    n_members=training_members,
    loss_func=CRPSLoss(),
    stride=data_options["stride"],
    optimizer_config=optimizer,
)

## Train with Lightning

Ten short CPU epochs exercise the workflow. Increase the data and training budget for a useful forecast model.

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
    accelerator="cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)
trainer.fit(model, datamodule=datamodule)
model.eval();

## See where randomness enters

Calling the processor directly lets you supply `x_noise`. Holding it fixed gives identical outputs. The ensemble wrapper instead calls `processor.map`, which draws fresh noise for each member at each forecast step.

In [ ]:
encoded, global_cond = encoder_decoder.encoder.encode_with_cond(batch)
noise = torch.randn(encoded.shape[0], noise_channels)
with torch.no_grad():
    fixed = processor(encoded, x_noise=noise, global_cond=global_cond)
    repeated = processor(encoded, x_noise=noise, global_cond=global_cond)
torch.testing.assert_close(fixed, repeated)

## Sample and plot an ensemble

Change `seed` or `n_members` and rerun these two cells without retraining. Each member follows its own autoregressive trajectory. Outputs have shape `(batch, time, height, width, channels, members)`.

In [ ]:
seed = 7
n_members = 5
rollout_batch = next(iter(datamodule.rollout_test_dataloader(batch_size=1)))
torch.manual_seed(seed)
with torch.no_grad():
    predictions, truth = model.rollout(
        rollout_batch,
        stride=data_options["stride"],
        max_rollout_steps=2,
        n_members=n_members,
        free_running_only=True,
    )
predictions.shape

In [ ]:
ensemble_mean = predictions.mean(dim=-1)
ensemble_spread = predictions.std(dim=-1, unbiased=False)
figure = plot_spatiotemporal_snapshots(
    true=truth,
    pred=ensemble_mean,
    pred_uq=ensemble_spread,
    extra_preds=[
        (predictions[..., i], f"Member {i + 1}") for i in range(min(2, n_members))
    ],
    timesteps=(0, 1, 3),
    pred_label="Ensemble mean",
    pred_uq_label="Member std",
    title="Noise-conditioned ensemble",
)
plt.show()

Spread shows that noise conditioning is active; it does not establish calibration. The difference row is the error in the ensemble mean.

## Save for evaluation

Save the checkpoint and construction settings so [evaluation and results](evaluation_and_results.ipynb) can compare this Python-trained model with diffusion and flow matching. That notebook covers CRPS, coverage and spread–skill behaviour.

In [ ]:
trainer.save_checkpoint(run_dir / "encoder_processor_decoder.ckpt")
config = OmegaConf.create(
    {
        "datamodule": {
            "_target_": "autocast.data.datamodule.SpatioTemporalDataModule",
            **data_options,
        },
        "model": {
            "encoder": {
                "_target_": "autocast.encoders.permute_concat.PermuteConcat",
                **encoder_options,
            },
            "decoder": {
                "_target_": "autocast.decoders.channels_last.ChannelsLast",
                **decoder_options,
            },
            "processor": {
                "_target_": "autocast.processors.azula_vit.AzulaViTProcessor",
                **processor_options,
            },
            "n_members": training_members,
            "loss_func": {"_target_": "autocast.losses.ensemble.CRPSLoss"},
        },
        "optimizer": optimizer,
        "output": {
            "save_config": True,
            "checkpoint_name": "encoder_processor_decoder.ckpt",
        },
    }
)
OmegaConf.save(config, run_dir / "resolved_config.yaml")